In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pdfplumber faiss-cpu langchain-upstage sentencepiece -q

import os, json
import pdfplumber
from tqdm import tqdm

In [3]:
# ==========================
# 1) PDF → TEXT 추출
# ==========================

import os
import pdfplumber

def extract_text_from_pdf(pdf_path: str) -> str:
    """PDF 전체 텍스트 추출"""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"{pdf_path} not found: 현재 경로 = {os.getcwd()}")

    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            txt = page.extract_text() or ""
            txt = txt.replace("\u00A0", " ")
            lines.append(txt)
        print(f"총 {len(pdf.pages)} 페이지에서 텍스트 추출 완료")

    return "\n".join(lines)

In [4]:
base = "/content/drive/MyDrive/rag-mmlu-ewha/data/external_kb"

domains = {
    "law": [
        f"{base}/외부문서(Law)_introlaw.pdf",
        f"{base}/외부문서(Law)_OpenCourseWare.pdf"
    ],

    "psychology": [
        f"{base}/외부문서(History)_openstaxworldhistory.pdf"
    ],

    "business": [
        f"{base}/외부문서(Business)_Marketing.pdf",
        f"{base}/외부문서(Business)_OpenStaxEconomics.pdf"
    ],

    "philosophy": [
        f"{base}/외부문서(Philosophy)_철학개론.pdf",
        f"{base}/외부문서(Philosophy)_OpenStaxSummary.pdf"
    ],

    "history": [
        f"{base}/외부문서(History)_openstaxworldhistory.pdf"
    ]
}

# 실제 PDF에서 텍스트 뽑아서 변수에 담기
documents = []   # [(domain, text), ...]

for domain, filelist in domains.items():
    for pdf_path in filelist:
        print(f"Processing [{domain}] {pdf_path}")
        text = extract_text_from_pdf(pdf_path)
        documents.append((domain, text))

Processing [law] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Law)_introlaw.pdf
총 32 페이지에서 텍스트 추출 완료
Processing [law] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Law)_OpenCourseWare.pdf
총 27 페이지에서 텍스트 추출 완료
Processing [psychology] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(History)_openstaxworldhistory.pdf
총 64 페이지에서 텍스트 추출 완료
Processing [business] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Business)_Marketing.pdf
총 10 페이지에서 텍스트 추출 완료
Processing [business] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Business)_OpenStaxEconomics.pdf
총 58 페이지에서 텍스트 추출 완료
Processing [philosophy] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Philosophy)_철학개론.pdf
총 8 페이지에서 텍스트 추출 완료
Processing [philosophy] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Philosophy)_OpenStaxSummary.pdf
총 27 페이지에서 텍스트 추출 완료
Processing [history] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(History)_openstaxworldhist

In [5]:
from collections import Counter

domain_counts = Counter([d for d, _ in documents])
print("도메인별 문서 개수:", domain_counts)

print("=== documents 샘플 2개 출력 ===")
for i in range(min(2, len(documents))):
    domain, text = documents[i]
    print(f"[{i}] domain = {domain}")
    print(text[:500], "...")  # 앞 500자만 미리보기
    print("=" * 80)

print("=== 각 문서 word 길이 확인 ===")
for i, (domain, text) in enumerate(documents):
    word_count = len(text.split())
    print(f"[{i}] domain={domain} | words={word_count}")

도메인별 문서 개수: Counter({'law': 2, 'business': 2, 'philosophy': 2, 'psychology': 1, 'history': 1})
=== documents 샘플 2개 출력 ===
[0] domain = law

Established in 1931, the Institute of Government provides training, advisory, and research
services to public officials and others interested in the operation of state and local
government in North Carolina. A part of The University of North Carolina at Chapel Hill,
the Institute also administers the university’s Master of Public Administration Program.
Each year approximately 14,000 city, county, and state officials attend one or more of the
230 classes, seminars, and conferences offered by th ...
[1] domain = law







Karl Marx (1818-1883). From the Preface to A Contribution to the Critique of Political
Economy. (1859) 1913, pp. 11-13.
Herbert, Bob. “Nike's Boot Camps.” In America. In New York Times. 31 Mar. 1997.
Gayle Krishenbanm. “Nike's Nemesis." In Newsmaker: Cicih Sukaesih.
Griswold, Wendy. "The Ideas of the Reading Class." Contemporary S

In [6]:
# TEXT → CHUNKS 변환

# 300 tokens(=약 200~300자) 전후가 RAG에서 가장 많이 쓰는 sweet spot
# 250~350 token chunk
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

kb = []
doc_id = 1

for domain, text in documents:
    chunks = chunk_text(text)
    for c in chunks:
        kb.append({
            "doc_id": f"{domain}-{doc_id}",
            "domain": domain,
            "source": "external_kb",
            "text": c
        })
        doc_id += 1

output_jsonl = "/content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl"

with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in kb:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_jsonl)
print("총 chunks:", len(kb))

'''
외부문서는 LangChain LLM cleanup 안한 이유
-> 성능 측면에서도 chunk만 하는 걸 추천
RAG에서는 "잘 정제된 문장"보다 "많은 coverage"가 더 중요
'''

Saved: /content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl
총 chunks: 267


'\n외부문서는 LangChain LLM cleanup 안한 이유\n-> 성능 측면에서도 chunk만 하는 걸 추천\nRAG에서는 "잘 정제된 문장"보다 "많은 coverage"가 더 중요\n'

In [7]:
output_jsonl = "/content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl"

with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in kb:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_jsonl)
print("총 chunks:", len(kb))

Saved: /content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl
총 chunks: 267


In [11]:
# ==========================
# 5) Upstage 임베딩 모델 정의
#    - 규정: 오픈소스 임베딩 대신 Upstage 사용
# ==========================

from google.colab import userdata
os.environ["UPSTAGE_API_KEY"] = userdata.get('UPSTAGE_API_KEY')

import numpy as np
from langchain_upstage import UpstageEmbeddings

class UpstageEmbeddingModel:
    def __init__(self, model_name: str = "solar-embedding-1-large"):
        self.model = UpstageEmbeddings(model=model_name)
        self.dim = None  # 임베딩 차원 기록용

    def encode_documents(self, texts, batch_size: int = 32) -> np.ndarray:
        """
        외부 KB chunk들을 임베딩.
        - texts: chunk 텍스트 리스트
        - 결과: (num_chunks, dim) numpy 배열
        """
        all_vecs = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            emb = self.model.embed_documents(batch)  # List[List[float]]
            all_vecs.extend(emb)

        arr = np.array(all_vecs, dtype="float32")
        print("Embedding 완료, shape(before norm):", arr.shape)

        # L2 정규화 → IP = cosine similarity
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        arr = arr / np.clip(norms, 1e-12, None)

        if self.dim is None:
            self.dim = arr.shape[1]
        elif self.dim != arr.shape[1]:
            raise ValueError(f"임베딩 차원 불일치: {self.dim} vs {arr.shape[1]}")

        print("첫 5개 벡터 L2 norm:", np.linalg.norm(arr[:5], axis=1))
        return arr

    def encode_query(self, text: str) -> np.ndarray:
        """
        나중에 retrieval 테스트할 때 사용할 쿼리 임베딩용.
        """
        vec = self.model.embed_query(text)
        arr = np.array(vec, dtype="float32").reshape(1, -1)

        norm = np.linalg.norm(arr, axis=1, keepdims=True)
        arr = arr / np.clip(norm, 1e-12, None)

        if self.dim is not None and arr.shape[1] != self.dim:
            raise ValueError(
                f"쿼리 차원({arr.shape[1]})과 코퍼스 차원({self.dim})이 다릅니다."
            )

        return arr

# ==========================
# 6) FAISS 인덱스 생성/저장
# ==========================

import faiss
import os

class FaissVectorStore:
    def __init__(self, dim: int, index_path: str):
        self.dim = dim
        self.index_path = index_path
        self.index = faiss.IndexFlatIP(dim)  # IP + 정규화 = cosine

    def build_index(self, vectors: np.ndarray):
        print("FAISS 인덱스 생성 중, shape:", vectors.shape)
        self.index.add(vectors)
        print("인덱스에 저장된 벡터 수:", self.index.ntotal)

    def save(self):
        os.makedirs(os.path.dirname(self.index_path), exist_ok=True)
        faiss.write_index(self.index, self.index_path)
        print("인덱스 저장 완료:", self.index_path)

    def load(self):
        if not os.path.exists(self.index_path):
            raise FileNotFoundError(f"인덱스 파일이 없습니다: {self.index_path}")
        self.index = faiss.read_index(self.index_path)
        print("인덱스 로드 완료, 벡터 수:", self.index.ntotal)

    def search(self, q_emb: np.ndarray, top_k: int = 5):
        if q_emb.ndim != 2:
            raise ValueError(f"q_emb는 2차원이어야 합니다. 현재 shape={q_emb.shape}")
        if q_emb.shape[1] != self.index.d:
            raise ValueError(f"쿼리 차원 {q_emb.shape[1]} != 인덱스 차원 {self.index.d}")
        D, I = self.index.search(q_emb, top_k)
        return D[0], I[0]

# ==========================
# 7) MMLU KB 임베딩 + 인덱스 생성 실행부
# ==========================

# kb 리스트는 이미 위에서 만든 상태라고 가정
texts = [row["text"] for row in kb]

embedder = UpstageEmbeddingModel()
print("외부 KB 임베딩 생성 시작")
vectors = embedder.encode_documents(texts, batch_size=32)
print("최종 vectors.shape:", vectors.shape)

index_path = "/content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb_index.faiss"

store = FaissVectorStore(dim=vectors.shape[1], index_path=index_path)
store.build_index(vectors)
store.save()


외부 KB 임베딩 생성 시작
Embedding 완료, shape(before norm): (267, 4096)
첫 5개 벡터 L2 norm: [1. 1. 1. 1. 1.]
최종 vectors.shape: (267, 4096)
FAISS 인덱스 생성 중, shape: (267, 4096)
인덱스에 저장된 벡터 수: 267
인덱스 저장 완료: /content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb_index.faiss


In [12]:
import os
import json
import numpy as np
import faiss
from langchain_upstage import UpstageEmbeddings

# 경로 설정
BASE_PATH = "/content/drive/MyDrive/rag-mmlu-ewha/data"
KB_PATH = os.path.join(BASE_PATH, "mmlu_kb.jsonl")
INDEX_PATH = os.path.join(BASE_PATH, "mmlu_kb_index.faiss")

# 1) KB 로딩 (텍스트 + 도메인)
kb_texts = []
kb_domains = []

with open(KB_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        kb_texts.append(obj["text"])
        kb_domains.append(obj["domain"])

print("KB 로드 완료. 총 chunk 개수:", len(kb_texts))

# 2) FAISS 인덱스 로드
index = faiss.read_index(INDEX_PATH)
print("FAISS 인덱스 정보:")
print(" - 차원(dim):", index.d)
print(" - 벡터 수:", index.ntotal)

# 3) Upstage 임베딩 초기화 (쿼리용)
embed_model = UpstageEmbeddings(model="solar-embedding-1-large")

def encode_query(text: str) -> np.ndarray:
    """쿼리 임베딩 + L2 정규화"""
    v = embed_model.embed_query(text)
    arr = np.array(v, dtype="float32").reshape(1, -1)
    norm = np.linalg.norm(arr, axis=1, keepdims=True)
    arr = arr / np.clip(norm, 1e-12, None)
    return arr

# 4) 샘플 쿼리로 검색 테스트
sample_queries = [
    "What is mens rea in criminal law?",
    "What is the marketing mix in business?",
    "What is utilitarianism in philosophy?"
]

for q in sample_queries:
    print("\n[질문]", q)
    q_vec = encode_query(q)
    print(" - q_vec.shape:", q_vec.shape)

    if q_vec.shape[1] != index.d:
        print("차원 불일치: 쿼리 dim =", q_vec.shape[1], ", 인덱스 dim =", index.d)
        continue

    scores, idxs = index.search(q_vec, 5)
    print(" - raw scores:", scores[0])

    print("=== TOP 5 결과 ===")
    for rank, i in enumerate(idxs[0]):
        print(f"[{rank+1}] score={scores[0][rank]:.4f}, domain={kb_domains[i]}")
        print(kb_texts[i][:300].replace("\n", " "))
        print("-" * 60)


KB 로드 완료. 총 chunk 개수: 267
FAISS 인덱스 정보:
 - 차원(dim): 4096
 - 벡터 수: 267

[질문] What is mens rea in criminal law?
 - q_vec.shape: (1, 4096)
 - raw scores: [0.3351946  0.32315847 0.3166545  0.3132726  0.31185082]
=== TOP 5 결과 ===
[1] score=0.3352, domain=law
to find the defendant guilty of assault inflicting serious injury, the State must prove two things beyond a reasonable doubt: First, that the defendant assaulted Wilbur Creecy by intention- ally and without justification and excuse hitting him with his fists. And second, that the defendant inflicted
------------------------------------------------------------
[2] score=0.3232, domain=law
breaking and entering took place at night. We also rely very heavily on common law for definitions of offenses such as “attempting” a crime, “conspiracy,” and “murder.” Common law definitions and court decisions revising common law determine in part how such crimes are charged and prosecuted. Common
------------------------------------------------------